
# P4 extra: entrenamiento de RT-DETR-L en Google Colab

**Uso:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar*, y después *Entorno de ejecución → Ejecutar todas*.

La celda hace todo sola:

1. Conecta Google Drive. Ahí se guarda el entrenamiento después de cada época, así que no se pierde aunque Colab se desconecte.
2. Clona el repositorio, que incluye el dataset `P4/dataset_matriculas`, e instala Ultralytics.
3. Entrena RT-DETR-L con la misma configuración que el YOLO11n de `VC_P4.ipynb`: `imgsz=640`, 40 épocas, `patience=10` y el mismo reparto de datos.
   Si ya había un entrenamiento a medias en Drive, **lo continúa** desde la última época.
4. Al terminar, guarda `matriculas_rtdetr_l.zip` en la carpeta `VC_P4_RTDETR` de Drive y lo descarga en el ordenador.
   Hay que copiar su contenido en `P4/runs/detect/matriculas_rtdetr_l/`.


In [ ]:

import os, sys, glob, shutil, subprocess, time

# ---------------------------------------------------------------- 1. Google Drive (para no perder el progreso)
from google.colab import drive, files
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/VC_P4_RTDETR'
os.makedirs(DRIVE_DIR, exist_ok=True)

# ---------------------------------------------------------------- 2. Repositorio (con el dataset) y Ultralytics
REPO = '/content/VC'
if not os.path.isdir(os.path.join(REPO, 'P4', 'dataset_matriculas')):
    print('Clonando el repositorio (~1 GB, tarda 1-2 minutos)...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Pegasos2005/VC.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.212'], check=True)

import torch
from ultralytics import RTDETR
assert torch.cuda.is_available(), ('No hay GPU. Ve a "Entorno de ejecución -> Cambiar tipo de entorno de ejecución", '
                                   'elige "T4 GPU", guarda y vuelve a ejecutar.')
print('GPU:', torch.cuda.get_device_name(0))

DATA_DIR = os.path.join(REPO, 'P4', 'dataset_matriculas')
YAML = os.path.join(DATA_DIR, 'matriculas_colab.yaml')
with open(YAML, 'w') as f:
    f.write(f"path: {DATA_DIR}\ntrain: train/images\nval: val/images\ntest: test/images\n\nnc: 1\nnames: ['matricula']\n")
for s in ('train', 'val', 'test'):
    print(f"{s:5s}: {len(glob.glob(os.path.join(DATA_DIR, s, 'images', '*')))} imágenes")

# ---------------------------------------------------------------- 3. Entrenamiento (continúa si se cortó)
NOMBRE  = 'matriculas_rtdetr_l'
RUN_DIR = os.path.join(DRIVE_DIR, NOMBRE)
LAST    = os.path.join(RUN_DIR, 'weights', 'last.pt')
FIN     = os.path.join(RUN_DIR, 'TERMINADO.txt')

t0 = time.time()
if os.path.exists(FIN):
    print('El entrenamiento ya había terminado; solo se vuelve a generar el zip.')
elif os.path.exists(LAST):
    print('Continuando el entrenamiento desde', LAST)
    RTDETR(LAST).train(resume=True)
else:
    RTDETR('rtdetr-l.pt').train(
        data=YAML, epochs=40, imgsz=640, batch=8, device=0, workers=2, cache=True,
        patience=10, seed=0, deterministic=False, plots=True,
        project=DRIVE_DIR, name=NOMBRE, exist_ok=True)
if not os.path.exists(FIN):
    with open(FIN, 'w') as f:
        f.write(f'Entrenamiento terminado. GPU: {torch.cuda.get_device_name(0)}. '
                f'Duración de esta sesión: {(time.time() - t0) / 60:.0f} min\n')

# ---------------------------------------------------------------- 4. Zip con los resultados (Drive + descarga)
zip_path = shutil.make_archive(os.path.join(DRIVE_DIR, NOMBRE), 'zip', RUN_DIR)
print(f'\n¡HECHO! Resultados en Google Drive: Mi unidad/VC_P4_RTDETR/{NOMBRE}.zip')
files.download(zip_path)
